# Задача 8. Обучаем сегментацию

Цель: обучить U-Net для бинарной сегментации человека на небольшом датасете, сравнить базовый подход, улучшения без смены архитектуры, аугментацию при инференсе, изменение архитектуры, ансамбль по пяти фолдам и U-Net с предобученным энкодером.

Ноутбук рассчитан на запуск в Colab с GPU. Все случайные генераторы фиксируются, метрика качества — Jaccard index.

## 1. Подготовка

In [ ]:
# mypy: ignore-errors
# ruff: noqa: E402, F401, F811

import importlib.util
import logging
import os
import subprocess
import sys
import warnings

IN_COLAB = "google.colab" in sys.modules

# Не пытаемся неявно читать HF_TOKEN из Colab Secrets: веса ImageNet
# можно скачать без авторизации, а проверка токена иногда даёт шумный warning.
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    module=r"huggingface_hub\.utils\._auth",
)
warnings.filterwarnings(
    "ignore",
    message=r"(?s).*HF_TOKEN.*",
    category=UserWarning,
    module=r"huggingface_hub\..*",
)
warnings.filterwarnings(
    "ignore",
    message=r"(?s).*unauthenticated requests to the HF Hub.*",
    category=UserWarning,
    module=r"huggingface_hub\..*",
)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

REQUIRED_PACKAGES = [
    "albumentations",
    "gdown",
    "matplotlib",
    "numpy",
    "opencv-python-headless",
    "pandas",
    "Pillow",
    "scikit-learn",
    "segmentation-models-pytorch",
    "torch",
    "torchvision",
    "tqdm",
]

if IN_COLAB:
    missing_packages = [
        package
        for package in REQUIRED_PACKAGES
        if importlib.util.find_spec(package.split("-")[0]) is None
    ]
    if missing_packages:
        install_command = [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            *missing_packages,
        ]
        subprocess.check_call(install_command)

In [ ]:
import gc
import random
import shutil
import zipfile
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Callable

import albumentations as A
import cv2
import gdown
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import segmentation_models_pytorch as smp
import torch
from IPython.display import display
from PIL import Image
from sklearn.model_selection import KFold, train_test_split
from torch import Tensor, nn, optim
from torch.utils.data import DataLoader, Dataset, Subset
from tqdm.auto import tqdm

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if DEVICE.type == "cpu":
    print(
        "GPU не найден. В Colab включите Среда выполнения -> Сменить тип "
        "среды выполнения -> GPU, "
        "иначе обучение будет идти на CPU и займет намного больше времени."
    )

DATASET_FILE_ID = "11wp4Bm-hEVwmZq8GmqbLpKpJiqvDSNAe"
DATA_DIR = Path("data/task_08_segmentation")
ARCHIVE_PATH = DATA_DIR / "dataset.zip"

IMAGE_SIZE = 192
BATCH_SIZE = 8
NUM_WORKERS = 0
TEST_SIZE = 0.2
VAL_SIZE = 0.2
TOP_N = 8

OVERFIT_STEPS = 350
BASELINE_EPOCHS = 12
IMPROVED_EPOCHS = 16
ARCH_EPOCHS = 12
FOLD_EPOCHS = 8
PRETRAINED_EPOCHS = 10

RUN_5FOLD_ENSEMBLE = True
RUN_PRETRAINED_ENCODER = True

print(f"device: {DEVICE}")

In [ ]:
def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)

## 2. Данные и разделение выборки

Датасет скачивается из Google Drive. Код ниже поддерживает структуру из практики `clip_img/.../*.jpg` и `matting/.../*.png`, а также более обычные структуры с папками `images` и `masks`.

In [ ]:
def download_dataset() -> None:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    has_images = any(DATA_DIR.rglob("*.jpg")) or any(DATA_DIR.rglob("*.png"))
    if has_images:
        print("dataset already exists")
        return

    url = f"https://drive.google.com/uc?id={DATASET_FILE_ID}"
    gdown.download(url, str(ARCHIVE_PATH), quiet=False, fuzzy=True)

    try:
        shutil.unpack_archive(str(ARCHIVE_PATH), str(DATA_DIR))
    except (shutil.ReadError, zipfile.BadZipFile):
        print(
            "Downloaded file is not a zip archive. "
            "Please unpack it into DATA_DIR manually."
        )
        raise


def infer_practice_mask_path(image_path: Path) -> Path | None:
    parts = list(image_path.parts)
    if "clip_img" not in parts:
        return None

    clip_idx = parts.index("clip_img")
    parts[clip_idx] = "matting"
    if len(parts) >= 2:
        parts[-2] = parts[-2].replace("clip", "matting")
    parts[-1] = f"{image_path.stem}.png"
    return Path(*parts)


def has_mask_token(path: Path) -> bool:
    mask_tokens = {"mask", "masks", "matting", "label", "labels"}
    path_tokens = {part.lower() for part in path.parts}
    return bool(mask_tokens & path_tokens)


def discover_pairs(root: Path) -> list[tuple[Path, Path]]:
    image_suffixes = {".jpg", ".jpeg", ".png"}
    all_images = [
        path
        for path in root.rglob("*")
        if (path.suffix.lower() in image_suffixes and not has_mask_token(path))
    ]

    mask_candidates = [
        path
        for path in root.rglob("*")
        if (path.suffix.lower() in image_suffixes and has_mask_token(path))
    ]
    masks_by_stem = {path.stem: path for path in mask_candidates}

    pairs = []
    for image_path in all_images:
        mask_path = infer_practice_mask_path(image_path)
        if mask_path is not None and mask_path.exists():
            pairs.append((image_path, mask_path))
            continue
        if image_path.stem in masks_by_stem:
            pairs.append((image_path, masks_by_stem[image_path.stem]))

    return sorted(set(pairs))


download_dataset()
pairs = discover_pairs(DATA_DIR)
print(f"found pairs: {len(pairs)}")
assert pairs, "No image/mask pairs found. " "Check DATA_DIR and dataset structure."
print(pairs[0])

In [ ]:
def read_image(path: Path) -> np.ndarray:
    image = Image.open(path).convert("RGB")
    return np.asarray(image)


def read_mask(path: Path) -> np.ndarray:
    mask_image = Image.open(path)
    if mask_image.mode == "RGBA":
        mask = np.asarray(mask_image.getchannel("A"))
    else:
        mask = np.asarray(mask_image.convert("L"))
    return (mask > 100).astype(np.float32)


def center_crop_person_area(
    image: np.ndarray,
    mask: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    if image.shape[0] > 220:
        image = image[100:-100]
        mask = mask[100:-100]
    return image, mask


class SelfieSegmentationDataset(Dataset):
    def __init__(
        self,
        image_mask_pairs: list[tuple[Path, Path]],
        image_size: int = IMAGE_SIZE,
        transform: A.Compose | None = None,
    ) -> None:
        self.image_mask_pairs = image_mask_pairs
        self.image_size = image_size
        self.transform = transform

    def __len__(self) -> int:
        return len(self.image_mask_pairs)

    def __getitem__(self, idx: int) -> dict[str, Tensor | str]:
        image_path, mask_path = self.image_mask_pairs[idx]
        image = read_image(image_path)
        mask = read_mask(mask_path)
        image, mask = center_crop_person_area(image, mask)

        image = cv2.resize(
            image,
            (self.image_size, self.image_size),
            interpolation=cv2.INTER_LINEAR,
        )
        mask = cv2.resize(
            mask,
            (self.image_size, self.image_size),
            interpolation=cv2.INTER_NEAREST,
        )

        if self.transform is not None:
            transformed = self.transform(image=image, mask=mask)
            image = transformed["image"]
            mask = transformed["mask"]

        image_tensor = torch.from_numpy(image.copy()).float().permute(2, 0, 1) / 255.0
        mask_tensor = torch.from_numpy(mask.copy()).float().unsqueeze(0)
        return {
            "image": image_tensor,
            "mask": mask_tensor,
            "path": str(image_path),
        }


def make_transforms(use_augmentations: bool) -> A.Compose | None:
    if not use_augmentations:
        return None
    return A.Compose(
        [
            A.HorizontalFlip(p=0.5),
            A.Affine(
                translate_percent={"x": (-0.05, 0.05), "y": (-0.05, 0.05)},
                scale=(0.88, 1.12),
                rotate=(-12, 12),
                border_mode=cv2.BORDER_CONSTANT,
                p=0.7,
            ),
            A.RandomBrightnessContrast(p=0.5),
            A.GaussNoise(p=0.2),
        ]
    )

In [ ]:
pair_indices = np.arange(len(pairs))
train_val_indices, test_indices = train_test_split(
    pair_indices,
    test_size=TEST_SIZE,
    random_state=SEED,
    shuffle=True,
)
train_indices, val_indices = train_test_split(
    train_val_indices,
    test_size=VAL_SIZE,
    random_state=SEED,
    shuffle=True,
)

print(
    "split sizes:",
    {
        "train": len(train_indices),
        "val": len(val_indices),
        "test": len(test_indices),
    },
)

base_dataset = SelfieSegmentationDataset(pairs, transform=None)
aug_dataset = SelfieSegmentationDataset(
    pairs,
    transform=make_transforms(True),
)

train_loader = DataLoader(
    Subset(base_dataset, train_indices.tolist()),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
aug_train_loader = DataLoader(
    Subset(aug_dataset, train_indices.tolist()),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    Subset(base_dataset, val_indices.tolist()),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    Subset(base_dataset, test_indices.tolist()),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

In [ ]:
sample = base_dataset[0]
plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.imshow(sample["image"].permute(1, 2, 0))
plt.title("image")
plt.axis("off")
plt.subplot(1, 2, 2)
plt.imshow(sample["mask"].squeeze(), cmap="gray")
plt.title("mask")
plt.axis("off")
plt.show()

## 3. Модель, Jaccard index и функции обучения

`BCEWithLogitsLoss` обучает бинарную маску по логитам. Jaccard index считается после бинаризации вероятностей по правилу `sigmoid(logits) > 0.5`.

In [ ]:
class CNNBlock(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 0,
    ) -> None:
        super().__init__()
        self.seq_block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size,
                stride,
                padding,
                bias=False,
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.seq_block(x)


class CNNBlocks(nn.Module):
    def __init__(
        self,
        n_conv: int,
        in_channels: int,
        out_channels: int,
        padding: int,
    ) -> None:
        super().__init__()
        layers = []
        for _ in range(n_conv):
            block = CNNBlock(in_channels, out_channels, padding=padding)
            layers.append(block)
            in_channels = out_channels
        self.layers = nn.Sequential(*layers)

    def forward(self, x: Tensor) -> Tensor:
        return self.layers(x)


class Encoder(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        padding: int,
        n_down: int = 4,
    ) -> None:
        super().__init__()
        layers: list[nn.Module] = []
        for _ in range(n_down):
            layers.extend(
                [
                    CNNBlocks(2, in_channels, out_channels, padding),
                    nn.MaxPool2d(2, 2),
                ]
            )
            in_channels = out_channels
            out_channels *= 2
        layers.append(CNNBlocks(2, in_channels, out_channels, padding))
        self.enc_layers = nn.ModuleList(layers)

    def forward(self, x: Tensor) -> tuple[Tensor, list[Tensor]]:
        connections = []
        for layer in self.enc_layers:
            if isinstance(layer, CNNBlocks):
                x = layer(x)
                connections.append(x)
            else:
                x = layer(x)
        return x, connections


class Decoder(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        exit_channels: int,
        padding: int,
        n_up: int = 4,
    ) -> None:
        super().__init__()
        layers: list[nn.Module] = []
        for _ in range(n_up):
            layers.extend(
                [
                    nn.ConvTranspose2d(
                        in_channels,
                        out_channels,
                        kernel_size=2,
                        stride=2,
                    ),
                    CNNBlocks(2, in_channels, out_channels, padding),
                ]
            )
            in_channels //= 2
            out_channels //= 2
        layers.append(nn.Conv2d(in_channels, exit_channels, kernel_size=1))
        self.layers = nn.ModuleList(layers)

    def forward(self, x: Tensor, connections: list[Tensor]) -> Tensor:
        skip_connections = connections[:-1]
        for layer in self.layers:
            if isinstance(layer, CNNBlocks):
                x = torch.cat([x, skip_connections.pop()], dim=1)
                x = layer(x)
            else:
                x = layer(x)
        return x


class UNET(nn.Module):
    def __init__(
        self,
        in_channels: int = 3,
        first_out_channels: int = 16,
        exit_channels: int = 1,
        n_down: int = 4,
        padding: int = 1,
    ) -> None:
        super().__init__()
        self.encoder = Encoder(
            in_channels,
            first_out_channels,
            padding,
            n_down,
        )
        self.decoder = Decoder(
            first_out_channels * (2**n_down),
            first_out_channels * (2 ** (n_down - 1)),
            exit_channels,
            padding,
            n_down,
        )

    def forward(self, x: Tensor) -> Tensor:
        enc_out, connections = self.encoder(x)
        return self.decoder(enc_out, connections)

In [ ]:
@dataclass
class EvalResult:
    loss: float
    jaccard: float
    per_sample: pd.DataFrame


PredictFn = Callable[[nn.Module, Tensor], Tensor]


def jaccard_from_logits(
    logits: Tensor,
    target: Tensor,
    threshold: float = 0.5,
) -> Tensor:
    probs = torch.sigmoid(logits)
    pred = probs > threshold
    target_bool = target > 0.5
    dims = tuple(range(1, pred.ndim))
    intersection = (pred & target_bool).sum(dim=dims).float()
    union = (pred | target_bool).sum(dim=dims).float()
    return (intersection + 1e-7) / (union + 1e-7)


def default_predict(model: nn.Module, x: Tensor) -> Tensor:
    return model(x)


def hflip_tta_predict(model: nn.Module, x: Tensor) -> Tensor:
    logits = model(x)
    flipped_logits = model(torch.flip(x, dims=[3]))
    flipped_logits = torch.flip(flipped_logits, dims=[3])
    return (logits + flipped_logits) / 2.0


def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    predict_fn: PredictFn = default_predict,
) -> EvalResult:
    model.eval()
    losses = []
    jaccard_indices = []
    rows = []

    with torch.no_grad():
        for batch in loader:
            x = batch["image"].to(DEVICE)
            y = batch["mask"].to(DEVICE)
            logits = predict_fn(model, x)
            loss = criterion(logits, y)
            batch_jaccard = jaccard_from_logits(logits, y)

            losses.append(loss.item())
            jaccard_indices.extend(batch_jaccard.detach().cpu().tolist())
            rows.extend(
                {
                    "path": path,
                    "jaccard": float(jaccard_index),
                }
                for path, jaccard_index in zip(
                    batch["path"],
                    batch_jaccard.detach().cpu().tolist(),
                    strict=False,
                )
            )

    return EvalResult(
        loss=float(np.mean(losses)),
        jaccard=float(np.mean(jaccard_indices)),
        per_sample=pd.DataFrame(rows),
    )


criterion = nn.BCEWithLogitsLoss()


def train_model(
    model: nn.Module,
    train_data: DataLoader,
    val_data: DataLoader,
    epochs: int,
    lr: float = 3e-4,
    weight_decay: float = 0.0,
    name: str = "model",
) -> tuple[nn.Module, pd.DataFrame]:
    model = model.to(DEVICE)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        train_losses = []
        train_jaccard_indices = []
        description = f"{name} epoch {epoch}/{epochs}"
        progress = tqdm(train_data, desc=description)
        for batch in progress:
            x = batch["image"].to(DEVICE)
            y = batch["mask"].to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            with torch.no_grad():
                train_losses.append(loss.item())
                batch_jaccard = jaccard_from_logits(logits, y)
                train_jaccard_indices.extend(batch_jaccard.cpu().tolist())
            progress.set_postfix(loss=float(np.mean(train_losses)))

        val_result = evaluate(model, val_data, criterion)
        history.append(
            {
                "epoch": epoch,
                "train_loss": float(np.mean(train_losses)),
                "train_jaccard": float(np.mean(train_jaccard_indices)),
                "val_loss": val_result.loss,
                "val_jaccard": val_result.jaccard,
                "name": name,
            }
        )

    return model, pd.DataFrame(history)


def free_gpu() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
def plot_history(history: pd.DataFrame, title: str) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(
        history["epoch"],
        history["train_loss"],
        label="train",
    )
    axes[0].plot(history["epoch"], history["val_loss"], label="val")
    axes[0].set_title(f"{title}: BCE loss")
    axes[0].set_xlabel("epoch")
    axes[0].legend()

    axes[1].plot(
        history["epoch"],
        history["train_jaccard"],
        label="train",
    )
    axes[1].plot(
        history["epoch"],
        history["val_jaccard"],
        label="val",
    )
    axes[1].set_title(f"{title}: Jaccard index")
    axes[1].set_xlabel("epoch")
    axes[1].legend()
    plt.show()


def show_predictions(
    model: nn.Module,
    loader: DataLoader,
    sample_paths: set[str] | None = None,
    limit: int = TOP_N,
    predict_fn: PredictFn = default_predict,
) -> None:
    model.eval()
    shown = 0
    with torch.no_grad():
        for batch in loader:
            paths = batch["path"]
            has_requested_path = sample_paths is None or any(
                path in sample_paths for path in paths
            )
            if not has_requested_path:
                continue

            x = batch["image"].to(DEVICE)
            y = batch["mask"]
            probs = torch.sigmoid(predict_fn(model, x)).cpu()

            for image, mask, prob, path in zip(
                batch["image"],
                y,
                probs,
                paths,
                strict=False,
            ):
                if sample_paths is not None and path not in sample_paths:
                    continue
                fig, axes = plt.subplots(1, 4, figsize=(14, 4))
                axes[0].imshow(image.permute(1, 2, 0))
                axes[0].set_title("image")
                axes[1].imshow(mask.squeeze(), cmap="gray")
                axes[1].set_title("target")
                axes[2].imshow(prob.squeeze(), cmap="magma")
                axes[2].set_title("probability")
                axes[3].imshow(prob.squeeze() > 0.5, cmap="gray")
                axes[3].set_title("prediction")
                for axis in axes:
                    axis.axis("off")
                plt.suptitle(Path(path).name)
                plt.show()
                shown += 1
                if shown >= limit:
                    return


def worst_paths(result: EvalResult, n: int = TOP_N) -> set[str]:
    return set(result.per_sample.nsmallest(n, "jaccard")["path"].tolist())

## 4. Проверка на переобучение одного батча

Если модель и разметка корректны, U-Net должен почти идеально выучить 4 изображения. После этого сравниваем Jaccard index на этом батче и на отложенной тестовой выборке.

In [ ]:
overfit_indices = train_indices[:4].tolist()
overfit_loader = DataLoader(
    Subset(base_dataset, overfit_indices),
    batch_size=4,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

overfit_model = UNET(first_out_channels=16).to(DEVICE)
overfit_criterion = nn.BCEWithLogitsLoss()
overfit_optimizer = optim.AdamW(overfit_model.parameters(), lr=1e-3)
fixed_batch = next(iter(overfit_loader))
fixed_x = fixed_batch["image"].to(DEVICE)
fixed_y = fixed_batch["mask"].to(DEVICE)
overfit_history = []

for step in tqdm(range(1, OVERFIT_STEPS + 1), desc="overfit one batch"):
    overfit_model.train()
    overfit_optimizer.zero_grad(set_to_none=True)
    overfit_logits = overfit_model(fixed_x)
    overfit_loss = overfit_criterion(overfit_logits, fixed_y)
    overfit_loss.backward()
    overfit_optimizer.step()

    if step == 1 or step % 25 == 0:
        overfit_jaccard = jaccard_from_logits(overfit_logits, fixed_y)
        overfit_history.append(
            {
                "step": step,
                "loss": overfit_loss.item(),
                "jaccard": float(overfit_jaccard.mean().item()),
            }
        )

overfit_history_df = pd.DataFrame(overfit_history)
display(overfit_history_df.tail())

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(overfit_history_df["step"], overfit_history_df["loss"])
plt.title("one batch loss")
plt.xlabel("step")
plt.subplot(1, 2, 2)
plt.plot(overfit_history_df["step"], overfit_history_df["jaccard"])
plt.title("one batch Jaccard")
plt.xlabel("step")
plt.show()

overfit_batch_result = evaluate(
    overfit_model,
    overfit_loader,
    overfit_criterion,
)
overfit_test_result = evaluate(
    overfit_model,
    test_loader,
    overfit_criterion,
)

display(
    pd.DataFrame(
        [
            {
                "experiment": "overfit batch",
                "split": "batch",
                "loss": overfit_batch_result.loss,
                "jaccard": overfit_batch_result.jaccard,
            },
            {
                "experiment": "overfit batch",
                "split": "test",
                "loss": overfit_test_result.loss,
                "jaccard": overfit_test_result.jaccard,
            },
        ]
    )
)

## 5. Базовый U-Net

Обучаем исходную архитектуру на обучающей выборке, подбираем количество эпох по максимуму `val_jaccard`, затем считаем качество на тестовой выборке. Оптимальное время обучения — эпоха с лучшим Jaccard index на валидационной выборке.

In [ ]:
set_seed(SEED)
baseline_model, baseline_history = train_model(
    UNET(first_out_channels=16),
    train_loader,
    val_loader,
    epochs=BASELINE_EPOCHS,
    lr=3e-4,
    name="baseline_unet",
)
plot_history(baseline_history, "baseline U-Net")

baseline_test = evaluate(baseline_model, test_loader, criterion)
best_baseline_idx = baseline_history["val_jaccard"].idxmax()
baseline_best_epoch = int(baseline_history.loc[best_baseline_idx, "epoch"])
print(f"Лучшая эпоха по Jaccard index на валидации: {baseline_best_epoch}")
print(f"Базовая модель, Jaccard index на тесте: {baseline_test.jaccard:.4f}")

In [ ]:
baseline_worst = worst_paths(baseline_test, TOP_N)
display(baseline_test.per_sample.sort_values("jaccard").head(TOP_N))

In [ ]:
show_predictions(
    baseline_model,
    test_loader,
    sample_paths=baseline_worst,
    limit=TOP_N,
)

## 6. Улучшение без смены архитектуры и TTA

Архитектура остаётся той же. Меняем только режим обучения: добавляем аугментации, оптимизатор AdamW, увеличиваем шаг обучения и добавляем L2-регуляризацию через `weight_decay`. Затем сравниваем обычный инференс на тестовой выборке и TTA с горизонтальным отражением.

In [ ]:
set_seed(SEED)
improved_model, improved_history = train_model(
    UNET(first_out_channels=16),
    aug_train_loader,
    val_loader,
    epochs=IMPROVED_EPOCHS,
    lr=1e-3,
    weight_decay=1e-4,
    name="augmented_unet",
)
plot_history(improved_history, "same U-Net + augmentations")

improved_test = evaluate(improved_model, test_loader, criterion)
improved_tta_test = evaluate(
    improved_model,
    test_loader,
    criterion,
    predict_fn=hflip_tta_predict,
)

display(
    pd.DataFrame(
        [
            {
                "experiment": "baseline",
                "test_jaccard": baseline_test.jaccard,
                "test_loss": baseline_test.loss,
            },
            {
                "experiment": "same architecture + augmentations",
                "test_jaccard": improved_test.jaccard,
                "test_loss": improved_test.loss,
            },
            {
                "experiment": ("same architecture + augmentations + hflip TTA"),
                "test_jaccard": improved_tta_test.jaccard,
                "test_loss": improved_tta_test.loss,
            },
        ]
    ).sort_values("test_jaccard", ascending=False)
)

## 7. Улучшение с изменением архитектуры

Проверяем более широкую U-Net: `first_out_channels=32`. Это меняет ёмкость модели, но сохраняет тот же общий encoder-decoder принцип.

In [ ]:
set_seed(SEED)
wide_model, wide_history = train_model(
    UNET(first_out_channels=32),
    aug_train_loader,
    val_loader,
    epochs=ARCH_EPOCHS,
    lr=7e-4,
    weight_decay=1e-4,
    name="wide_unet",
)
plot_history(wide_history, "wider U-Net")
wide_test = evaluate(wide_model, test_loader, criterion)
wide_tta_test = evaluate(
    wide_model,
    test_loader,
    criterion,
    predict_fn=hflip_tta_predict,
)
print(f"Широкий U-Net, Jaccard index на тесте: {wide_test.jaccard:.4f}")
print(f"Широкий U-Net + TTA, Jaccard index на тесте: {wide_tta_test.jaccard:.4f}")

## 8. Ансамбль по пяти фолдам внутри обучающей части

Тестовая выборка остаётся фиксированной. Внутри `train_val_indices` делаем пять фолдов, обучаем пять моделей, оцениваем каждую модель на тестовой выборке и усредняем логиты всех моделей до бинаризации и расчёта метрик.

In [ ]:
def make_loader(
    dataset: Dataset,
    indices: np.ndarray,
    batch_size: int = BATCH_SIZE,
    shuffle: bool = False,
) -> DataLoader:
    return DataLoader(
        Subset(dataset, indices.tolist()),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )


def ensemble_predict(models: list[nn.Module], x: Tensor) -> Tensor:
    logits = [model(x) for model in models]
    return torch.stack(logits).mean(dim=0)


fold_models = []
fold_histories = []
fold_test_results = []
fold_worst_sets: dict[str, set[str]] = {}

if RUN_5FOLD_ENSEMBLE:
    kfold = KFold(n_splits=5, shuffle=True, random_state=SEED)
    for fold, (fold_train_pos, fold_val_pos) in enumerate(
        kfold.split(train_val_indices),
        start=1,
    ):
        set_seed(SEED + fold)
        fold_train_indices = train_val_indices[fold_train_pos]
        fold_val_indices = train_val_indices[fold_val_pos]
        fold_train_loader = make_loader(
            aug_dataset,
            fold_train_indices,
            shuffle=True,
        )
        fold_val_loader = make_loader(
            base_dataset,
            fold_val_indices,
            shuffle=False,
        )

        fold_model, fold_history = train_model(
            UNET(first_out_channels=16),
            fold_train_loader,
            fold_val_loader,
            epochs=FOLD_EPOCHS,
            lr=1e-3,
            weight_decay=1e-4,
            name=f"fold_{fold}",
        )
        fold_test = evaluate(fold_model, test_loader, criterion)
        fold_models.append(fold_model)
        fold_histories.append(fold_history.assign(fold=fold))
        fold_test_results.append(
            {
                "experiment": f"fold_{fold}",
                "test_loss": fold_test.loss,
                "test_jaccard": fold_test.jaccard,
            }
        )
        fold_worst_sets[f"fold_{fold}"] = worst_paths(fold_test, TOP_N)

    all_fold_history = pd.concat(fold_histories, ignore_index=True)
    for fold, fold_history in all_fold_history.groupby("fold"):
        plt.plot(
            fold_history["epoch"],
            fold_history["val_jaccard"],
            label=f"fold {fold}",
        )
    plt.title("5-fold validation Jaccard")
    plt.xlabel("epoch")
    plt.legend()
    plt.show()

    ensemble_result = evaluate(
        fold_models[0],
        test_loader,
        criterion,
        predict_fn=lambda _model, x: ensemble_predict(fold_models, x),
    )
    fold_test_results.append(
        {
            "experiment": "5fold logits ensemble",
            "test_loss": ensemble_result.loss,
            "test_jaccard": ensemble_result.jaccard,
        }
    )
    fold_comparison = pd.DataFrame(fold_test_results).sort_values(
        "test_jaccard",
        ascending=False,
    )
else:
    all_fold_history = pd.DataFrame()
    ensemble_result = None
    fold_comparison = pd.DataFrame()

display(fold_comparison)

In [ ]:
if RUN_5FOLD_ENSEMBLE:
    bad_counter = Counter(path for paths in fold_worst_sets.values() for path in paths)
    repeated_bad_samples = pd.DataFrame(
        [
            {
                "path": path,
                "bad_in_folds": count,
                "file": Path(path).name,
            }
            for path, count in bad_counter.most_common()
        ]
    )
else:
    repeated_bad_samples = pd.DataFrame(columns=["path", "bad_in_folds", "file"])

display(repeated_bad_samples.head(20))

### Одна модель на всей обучающей части

Для честного сравнения на тестовой выборке обучаем одну модель на всех данных, кроме тестовых. Валидационную кривую для этого запуска не используем как независимую оценку, потому что часть `val` входит в обучение; качество сравниваем только на тестовой выборке.

In [ ]:
all_train_loader = make_loader(
    aug_dataset,
    train_val_indices,
    shuffle=True,
)
set_seed(SEED)
all_data_model, all_data_history = train_model(
    UNET(first_out_channels=16),
    all_train_loader,
    val_loader,
    epochs=IMPROVED_EPOCHS,
    lr=1e-3,
    weight_decay=1e-4,
    name="all_train_val_unet",
)
all_data_test = evaluate(all_data_model, test_loader, criterion)
print(
    "Одна модель на всей обучающей части, Jaccard index на тесте: "
    f"{all_data_test.jaccard:.4f}"
)

## 9. U-Net с предобученным энкодером

Используем `segmentation_models_pytorch.Unet` с энкодером `resnet18` и весами ImageNet. Архитектурно это уже другой энкодер, поэтому сравниваем отдельно.

In [ ]:
if RUN_PRETRAINED_ENCODER:
    set_seed(SEED)
    pretrained_model = smp.Unet(
        encoder_name="resnet18",
        encoder_weights="imagenet",
        in_channels=3,
        classes=1,
        activation=None,
    )
    pretrained_model, pretrained_history = train_model(
        pretrained_model,
        aug_train_loader,
        val_loader,
        epochs=PRETRAINED_EPOCHS,
        lr=3e-4,
        weight_decay=1e-4,
        name="resnet18_pretrained_unet",
    )
    plot_history(pretrained_history, "pretrained ResNet18 U-Net")
    pretrained_test = evaluate(pretrained_model, test_loader, criterion)
    pretrained_tta_test = evaluate(
        pretrained_model,
        test_loader,
        criterion,
        predict_fn=hflip_tta_predict,
    )
else:
    pretrained_history = pd.DataFrame()
    pretrained_test = None
    pretrained_tta_test = None

if pretrained_test is not None:
    print(f"Предобученный U-Net, Jaccard index на тесте: {pretrained_test.jaccard:.4f}")
    print(
        "Предобученный U-Net + TTA, Jaccard index на тесте: "
        f"{pretrained_tta_test.jaccard:.4f}"
    )

## 10. Сводная таблица и выводы

Итоговая таблица собирает все метрики на тестовой выборке. Текст ниже формируется из результатов текущего запуска, поэтому выводы не нужно переписывать вручную после повторного обучения.

In [ ]:
summary_rows = [
    {
        "experiment": "overfit batch evaluated on same 4 images",
        "test_loss": overfit_batch_result.loss,
        "test_jaccard": overfit_batch_result.jaccard,
    },
    {
        "experiment": "overfit batch evaluated on test",
        "test_loss": overfit_test_result.loss,
        "test_jaccard": overfit_test_result.jaccard,
    },
    {
        "experiment": "baseline U-Net",
        "test_loss": baseline_test.loss,
        "test_jaccard": baseline_test.jaccard,
    },
    {
        "experiment": "same U-Net + augmentations/hparams",
        "test_loss": improved_test.loss,
        "test_jaccard": improved_test.jaccard,
    },
    {
        "experiment": "same U-Net + augmentations/hparams + hflip TTA",
        "test_loss": improved_tta_test.loss,
        "test_jaccard": improved_tta_test.jaccard,
    },
    {
        "experiment": "wider U-Net",
        "test_loss": wide_test.loss,
        "test_jaccard": wide_test.jaccard,
    },
    {
        "experiment": "wider U-Net + hflip TTA",
        "test_loss": wide_tta_test.loss,
        "test_jaccard": wide_tta_test.jaccard,
    },
    {
        "experiment": "single U-Net trained on all train_val",
        "test_loss": all_data_test.loss,
        "test_jaccard": all_data_test.jaccard,
    },
]

if ensemble_result is not None:
    summary_rows.append(
        {
            "experiment": "5-fold logits ensemble",
            "test_loss": ensemble_result.loss,
            "test_jaccard": ensemble_result.jaccard,
        }
    )
if pretrained_test is not None:
    summary_rows.extend(
        [
            {
                "experiment": "pretrained ResNet18 U-Net",
                "test_loss": pretrained_test.loss,
                "test_jaccard": pretrained_test.jaccard,
            },
            {
                "experiment": "pretrained ResNet18 U-Net + hflip TTA",
                "test_loss": pretrained_tta_test.loss,
                "test_jaccard": pretrained_tta_test.jaccard,
            },
        ]
    )

summary = pd.DataFrame(summary_rows).sort_values(
    "test_jaccard",
    ascending=False,
)
display(summary)

In [ ]:
test_summary = summary[
    ~summary["experiment"].eq("overfit batch evaluated on same 4 images")
]
best_test_row = test_summary.iloc[0]
tta_delta = improved_tta_test.jaccard - improved_test.jaccard
aug_delta = improved_test.jaccard - baseline_test.jaccard
wide_delta = wide_test.jaccard - improved_test.jaccard
batch_jaccard_index = overfit_batch_result.jaccard
batch_test_gap = overfit_batch_result.jaccard - overfit_test_result.jaccard

print("Выводы")
print("======")
print(
    "1. Проверка на одном батче: "
    f"Jaccard index={batch_jaccard_index:.4f}. "
    "Если значение близко к 1, модель и пайплайн разметки работают "
    "корректно. "
    f"Разрыв с тестовой выборкой равен {batch_test_gap:.4f}, "
    "что ожидаемо для переобучения на 4 картинках."
)
print(
    "2. Для базового U-Net лучшая эпоха по Jaccard index на валидации: "
    f"{baseline_best_epoch}. "
    "При полном запуске после этой точки стоит останавливать обучение "
    "или использовать раннюю остановку."
)
print(
    "3. Аугментации и гиперпараметры изменили Jaccard index на тестовой "
    f"выборке на {aug_delta:+.4f} относительно базовой модели. "
    "При коротком запуске на 1 эпоху знак дельты показывает только "
    "результат текущего эксперимента, а не устойчивое преимущество метода."
)
print(
    "4. TTA с горизонтальным отражением изменил Jaccard index на тестовой "
    f"выборке на {tta_delta:+.4f}. Малый эффект означает, что отражение "
    "почти не меняет предсказания текущей модели."
)
print(
    "5. Более широкая архитектура изменила Jaccard index на тестовой "
    f"выборке на {wide_delta:+.4f} относительно улучшенной модели "
    "без смены архитектуры."
)
if ensemble_result is not None:
    ensemble_delta = ensemble_result.jaccard - all_data_test.jaccard
    repeated_count = int((repeated_bad_samples["bad_in_folds"] > 1).sum())
    print(
        "6. Ансамбль по пяти фолдам отличается от одной модели, обученной "
        "на всей обучающей части, на "
        f"{ensemble_delta:+.4f} Jaccard index. Среди "
        f"top-{TOP_N} худших предсказаний по фолдам повторяются "
        f"{repeated_count} семплов; частые повторы указывают "
        "на системно сложные изображения."
    )
if pretrained_test is not None:
    pretrained_delta = pretrained_test.jaccard - baseline_test.jaccard
    print(
        "7. U-Net с предобученным ResNet18-энкодером изменил "
        "Jaccard index на тестовой выборке на "
        f"{pretrained_delta:+.4f} относительно базовой модели. "
        "На маленьком датасете предобученный энкодер часто помогает, "
        "если домен не слишком далёк от ImageNet."
    )
print(
    "Итог: среди запусков, оценённых на тестовой выборке, лучший результат — "
    f"{best_test_row['experiment']} с Jaccard index="
    f"{best_test_row['test_jaccard']:.4f}."
)